# Safe Health Query Chatbot using Prompt Engineering (LLM-Based)

**Author:** Muhammad Ali Waris Khan  

### 👨‍💻 Internship Project – AI/ML Engineering  

## 📌 Problem Statement

In today’s digital era, users frequently search for health-related information online. However, most sources are unverified, inconsistent, or potentially misleading.

To address this, we aim to build a **safe and controlled AI chatbot** that can respond to general health-related queries using a Large Language Model (LLM) with carefully designed prompt engineering.

The system is designed to:
- Answer general health questions in simple language
- Avoid medical diagnosis or prescriptions
- Ensure safe and responsible AI responses

---

## 🎯 Objective

The main objectives of this project are:

- To understand and implement prompt engineering techniques
- To interact with a Large Language Model
- To design safety-aware AI systems
- To build a simple conversational health assistant
- To ensure ethical constraints in AI-generated responses

# 1. ⚙️ Environment Setup

In this step, we set up the required libraries and API configuration to interact with a Large Language Model.

### Model Selection (Free LLM)

Since we are not using OpenAI API, we will use a free instruction-tuned model from Hugging Face.

We choose:

- **Model:** `google/flan-t5-base`

### 📌 Why this model?
- Instruction-tuned (understands prompts well)
- Lightweight (runs in Colab)
- Good for Q&A style tasks
- Free and widely used for educational LLM projects

This model will act as the core engine of our chatbot.

In [12]:
!pip install transformers sentencepiece

## 2. Model Loading

Due to compatibility issues with the Hugging Face pipeline API in newer versions, we directly use the model and tokenizer for inference.

This approach is:
- More stable across library versions
- Closer to production-level implementation
- Gives full control over generation behavior

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### 📌 Model Explanation

We use a pre-trained instruction-tuned model (FLAN-T5).

Unlike simple language models, this model:
- Understands direct instructions
- Generates structured responses
- Works well for question-answering tasks

This makes it suitable for building a basic health query chatbot.

## 3. Prompt Engineering System

Prompt engineering is the process of designing structured inputs that guide a language model to produce safe, relevant, and controlled outputs.

In this project, we design a carefully constrained prompt to ensure that the chatbot behaves responsibly when answering health-related questions.

---

### 🎯 Design Goals

The prompt is designed to achieve the following:

- Provide general health information only
- Prevent medical diagnosis or prescriptions
- Encourage safe behavior (consulting professionals when needed)
- Maintain simple and user-friendly explanations

---

### ⚠️ Safety Constraints

To ensure responsible AI behavior, the model is explicitly instructed to:

- Avoid diagnosing diseases
- Avoid recommending medications or dosages
- Avoid definitive medical claims
- Redirect risky queries to professional healthcare advice when necessary

---

### Why Prompt Engineering Matters

Large Language Models are highly sensitive to input instructions. Even small changes in wording can significantly affect output behavior.

Therefore, prompt design acts as a **control layer** that shapes model behavior without modifying the model itself.

In [14]:
def generate_health_response(user_query):

    prompt = f"""
You are a safe, responsible, and helpful health information assistant.

Strict rules:
- Do NOT diagnose medical conditions
- Do NOT prescribe medicines or dosages
- Do NOT give emergency treatment instructions
- Provide only general educational health information
- If the question involves risk, clearly suggest consulting a doctor

User Question: {user_query}

Provide a clear, simple, and safe response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=200,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response.strip()

### 📌 Prompt Design Explanation

We define a structured prompt that guides the model behavior.

Key components:

- **Role definition:** The model acts as a health information assistant
- **Safety rules:** Prevents medical diagnosis or prescriptions
- **Output constraint:** Ensures simple and safe responses

This is a core concept in LLM applications called **instruction prompting**.

## 4. 💬 Testing the Chatbot

We now test the system with sample user queries to evaluate response quality and safety behavior.

In [15]:
test_queries = [
    "What causes fever?",
    "Can I take paracetamol for headache?",
    "How to reduce anxiety naturally?",
    "What should I do if I have chest pain?",
    "Is sleeping 5 hours enough?"
]

for q in test_queries:
    print("\n🧑 User:", q)
    print("🤖 Bot:", generate_health_response(q))
    print("-"*60)


🧑 User: What causes fever?
🤖 Bot: viral infection in body The fever itself occurs during symptoms which do something
------------------------------------------------------------

🧑 User: Can I take paracetamol for headache?
🤖 Bot: Some sources mention getting rid migraine headache : A few reviews recommend parisic and paraxaine on paracet'am' that has strong to reduce burning muscle strain for severe and mild-inquises among men taking antihisticand drugs are given paricellom that was prescribed twice more f for moderate (trepiandilis in dogs used too short. it usually cure adults after 2 dose) although in studies adults still fail most effective when recommended above with parcocetaxemus can no overprotect an adult as is shown too low that many patients have experienced moderate levels under 1 to 2 months duration in what symptoms usually manifest to have the typical mild of it before treatment gets out it is just more like painful burning after using. most physicians provide more eff

## 5. 📊 Model Output Analysis & Interpretation

In this section, we analyze the behavior of the chatbot based on real test queries.

---

### ⚠️ Observed Issue: Response Instability

During testing, the model produced:

- Irrelevant and incoherent outputs
- Repetitive or nonsensical text
- Hallucinated medical statements
- Weak adherence to prompt instructions

This indicates that the model is not consistently following the safety constraints defined in the prompt.

---

### Why This Happens

This behavior is expected due to several limitations:

#### 1. Small Model Capacity
FLAN-T5-base is a lightweight model and is not optimized for strict instruction-following in sensitive domains like healthcare.

#### 2. No Domain Fine-Tuning
The model is not specifically trained on medical or safety-critical dialogue datasets.

#### 3. Limited Safety Alignment
Unlike commercial LLMs, open-source models do not have strong reinforcement learning from human feedback (RLHF).

#### 4. Prompt Sensitivity
The model is highly sensitive to prompt wording and may ignore constraints under certain inputs.

---

### 📌 Key Insight

Even with well-designed prompts, model behavior is ultimately constrained by the underlying training quality and architecture.

This demonstrates an important real-world AI concept:

> **Prompt engineering can guide behavior, but it cannot fully override model limitations.**

---

### ⚠️ Safety Concern

Some outputs contained:
- Irrelevant medical claims
- Potentially misleading or unsafe information

Therefore, this system should NOT be used in real healthcare scenarios.

## 6. 🔧 System Improvement: Safety Layer Integration

To improve reliability, we introduced two major enhancements:

### 1. Input Filtering Layer
Detects potentially risky medical queries and prevents them from reaching the model.

### 2. Deterministic Decoding
We disabled sampling (do_sample=False) to reduce randomness and hallucination.

---

### 📌 Result of Improvements

These changes improve:
- Response stability
- Safety compliance
- Predictability of outputs

However, limitations of the base model still remain.

In [16]:
def is_safe_input(text):
    risky_keywords = [
        "dose", "medicine", "drug", "overdose",
        "treatment", "prescribe", "suicide"
    ]

    text_lower = text.lower()

    for word in risky_keywords:
        if word in text_lower:
            return False

    return True

In [17]:
def generate_health_response(user_query):

    if not is_safe_input(user_query):
        return "⚠️ This question may require professional medical guidance. Please consult a qualified healthcare provider."

    prompt = f"""
You are a safe health information assistant.

Rules:
- Only provide general educational information
- Never give medical diagnosis
- Never suggest medication or dosage
- Keep answers simple and safe
- If unsure, recommend consulting a doctor

Question: {user_query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=180,
        do_sample=False  # IMPORTANT FIX: removes randomness
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [18]:
test_queries = [
    "What causes fever?",
    "Can I take paracetamol for headache?",
    "How to reduce anxiety naturally?",
    "What should I do if I have chest pain?",
    "Is sleeping 5 hours enough?"
]

for q in test_queries:
    print("\n🧑 User:", q)
    print("🤖 Bot:", generate_health_response(q))
    print("-"*60)


🧑 User: What causes fever?
🤖 Bot: a cold
------------------------------------------------------------

🧑 User: Can I take paracetamol for headache?
🤖 Bot: No
------------------------------------------------------------

🧑 User: How to reduce anxiety naturally?
🤖 Bot: Try a few simple ways to reduce anxiety
------------------------------------------------------------

🧑 User: What should I do if I have chest pain?
🤖 Bot: If unsure, recommend consulting a doctor
------------------------------------------------------------

🧑 User: Is sleeping 5 hours enough?
🤖 Bot: If unsure, recommend consulting a doctor
------------------------------------------------------------


## 7. Final Conclusion

### 📌 Project Summary

In this project, we developed a **safe health-related question answering system** using a pre-trained open-source Large Language Model (FLAN-T5) combined with prompt engineering and rule-based safety filtering.

The objective was not to build a medical diagnostic system, but to explore how **LLMs can be controlled to generate safe and informative responses** using prompt design and simple system constraints.

---

### 📊 Key Outcomes

- Successfully implemented an end-to-end chatbot pipeline using a free LLM
- Designed a structured prompt to guide model behavior
- Added a safety filtering layer to handle risky queries
- Improved response stability using deterministic decoding
- Evaluated model behavior across general and sensitive health questions

---

### 🧠 Key Learnings

This project highlighted several important AI/ML concepts:

#### 1. Prompt Engineering is Powerful but Limited
Well-designed prompts improve behavior but cannot fully override model limitations.

#### 2. Model Choice Matters
Smaller models like FLAN-T5-base are not fully reliable for safety-critical domains.

#### 3. Safety Requires Multi-Layer Design
Real-world AI systems require:
- Input filtering
- Prompt constraints
- Controlled decoding strategies

#### 4. LLMs are Not Knowledge Bases
They generate probabilistic outputs and may still produce vague or overly generic responses.

---

### ⚠️ Limitations

- Model is not fine-tuned for medical or safety-critical domains
- Responses may be overly generic or inconsistent
- Rule-based filtering is limited and not exhaustive
- No real-world medical knowledge verification is applied

---

### 🚀 Future Improvements

- Fine-tuning on medical QA datasets
- Using instruction-tuned larger models (e.g., Mistral, LLaMA-based models)
- Adding retrieval-augmented generation (RAG) with trusted medical sources
- Improving safety classification using ML-based toxicity/risk detection models

---

### 📌 Final Takeaway

This project demonstrates how LLMs can be adapted into safe, controlled applications using prompt engineering and simple system design.

However, it also shows that **true reliability in healthcare AI requires stronger models, better alignment, and external knowledge grounding.**